# PixiJoy subject matting (M3) — Colab GPU

**Runtime → GPU → Run all.**

- Data: `prepare_quality_data.py` auto-downloads (no photo upload, **no TFDS**).
- Drive: **optional**. Default off. Turn `USE_DRIVE = True` only if you want
  checkpoints to survive disconnect / resume later.

Download `dist/matting.tflite` only if verify prints OK → [ml-packs-v1](https://github.com/Saikumar-S0906/Release-/releases/tag/ml-packs-v1).


In [ ]:
!nvidia-smi || true
import tensorflow as tf
print("TF", tf.__version__, "GPUs", tf.config.list_physical_devices("GPU"))


In [ ]:
# Drive is OPTIONAL — keep False unless you need resume after disconnect
USE_DRIVE = False

import os, urllib.request, zipfile, shutil
from pathlib import Path
from google.colab import files

ROOT = Path("/content/pixijoy_matting")
DRIVE = Path("/content/drive/MyDrive/pixijoy_ml/matting")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE.mkdir(parents=True, exist_ok=True)
    (DRIVE / "dist").mkdir(parents=True, exist_ok=True)
else:
    print("USE_DRIVE=False — ckpts in /content only (lost if runtime dies)")

ZIP_URL = "https://raw.githubusercontent.com/Saikumar-S0906/Release-/main/matting_lab.zip"
ok = False
try:
    urllib.request.urlretrieve(ZIP_URL, "matting_lab.zip")
    with zipfile.ZipFile("matting_lab.zip") as z:
        z.extractall(".")
    ok = Path("train.py").is_file()
    print("extracted lab from Release-", ok)
except Exception as e:
    print("zip fetch failed:", e)
if not ok:
    print("Upload matting_lab.zip")
    uploaded = files.upload()
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        z.extractall(".")
    ok = Path("train.py").is_file()
assert ok, "Could not load lab zip"

Path("dist").mkdir(parents=True, exist_ok=True)
if USE_DRIVE:
    # Point dist → Drive for resume
    local_dist = Path("dist")
    if local_dist.is_symlink() or local_dist.is_file():
        local_dist.unlink()
    elif local_dist.is_dir() and not any(local_dist.iterdir()):
        local_dist.rmdir()
    elif local_dist.is_dir():
        for p in local_dist.iterdir():
            shutil.copy2(p, DRIVE / "dist" / p.name) if p.is_file() else None
        shutil.rmtree(local_dist)
    if not Path("dist").exists():
        Path("dist").symlink_to(DRIVE / "dist")
    print("ckpt dir", DRIVE / "dist")
else:
    print("ckpt dir", Path("dist").resolve())


In [ ]:
TRAIN_MODE = "quality"
FORCE_FRESH = False

from pathlib import Path

assert Path("prepare_quality_data.py").is_file(), "lab zip missing prepare_quality_data.py — refresh Release-"
print("Fetching quality data (no TFDS, no manual photos)…")
rc = get_ipython().system("python -u prepare_quality_data.py --data ./data")
if rc != 0:
    raise SystemExit("prepare_quality_data failed")
assert (Path("data") / "READY").is_file()


In [ ]:
from pathlib import Path
USE_DRIVE = globals().get("USE_DRIVE", False)
OUT = (Path("/content/drive/MyDrive/pixijoy_ml/matting/dist/matting.keras")
       if USE_DRIVE else Path("dist") / "matting.keras")
OUT.parent.mkdir(parents=True, exist_ok=True)
ckpt = OUT.with_suffix(".ckpt.keras")
print("will resume" if ckpt.is_file() else "fresh train", ckpt)

flags = f'--out "{OUT}"'
flags += " --quality" if TRAIN_MODE == "quality" else " --smoke"
flags += " --no-resume" if FORCE_FRESH else ""
print("RUN python -u train.py", flags)
rc = get_ipython().system(f"python -u train.py {flags}")
if rc != 0:
    raise SystemExit("train failed")
rc = get_ipython().system(f'python -u export_tflite.py --keras "{OUT}" --out "dist/matting.tflite"')
if rc != 0:
    raise SystemExit("export failed")
vflags = "--quality" if TRAIN_MODE == "quality" else ""
rc = get_ipython().system(f"python -u verify_contract.py {vflags}")
if rc != 0:
    raise SystemExit("verify failed — do NOT pin")
import hashlib
p = Path("dist/matting.tflite")
assert p.is_file(), p
print("READY", p, p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest())


In [ ]:
from google.colab import files
files.download("dist/matting.tflite")
print("Upload to ml-packs-v1 as matting.tflite ONLY if verify OK")
